## Импорт библиотек

In [1]:
import pandas as pd
import sqlite3

## Подключение к базе

In [2]:
connection = sqlite3.connect("../data/checking-logs.sqlite")

## Структура таблицы

In [3]:
pd.io.sql.read_sql(
    "PRAGMA table_info(test);",
    connection
)

Out[3]: 
   cid             name       type  notnull dflt_value  pk
0    0              uid       TEXT        0       None   0
1    1          labname       TEXT        0       None   0
2    2  first_commit_ts  TIMESTAMP        0       None   0
3    3    first_view_ts  TIMESTAMP        0       None   0


## Обработка и анализ данных

In [4]:
pd.io.sql.read_sql(
    """
    SELECT *
    FROM test
    LIMIT 10;
    """,
    connection
)

Out[4]: 
       uid   labname             first_commit_ts               first_view_ts
0   user_1    laba04  2020-04-26 17:06:18.462708  2020-04-26 21:53:59.624136
1   user_1   laba04s  2020-04-26 17:12:11.843671  2020-04-26 21:53:59.624136
2   user_1    laba05  2020-05-02 19:15:18.540185  2020-04-26 21:53:59.624136
3   user_1    laba06  2020-05-17 16:26:35.268534  2020-04-26 21:53:59.624136
4   user_1   laba06s  2020-05-20 12:23:37.289724  2020-04-26 21:53:59.624136
5   user_1  project1  2020-05-14 20:56:08.898880  2020-04-26 21:53:59.624136
6  user_10    laba04  2020-04-25 08:24:52.696624  2020-04-18 12:19:50.182714
7  user_10   laba04s  2020-04-25 08:37:54.604222  2020-04-18 12:19:50.182714
8  user_10    laba05  2020-05-01 19:27:26.063245  2020-04-18 12:19:50.182714
9  user_10    laba06  2020-05-19 11:39:28.885637  2020-04-18 12:19:50.182714


## Структура таблицы

In [5]:
pd.io.sql.read_sql(
    "PRAGMA table_info(deadlines);",
    connection
)

Out[5]: 
   cid       name     type  notnull dflt_value  pk
0    0      index  INTEGER        0       None   0
1    1       labs     TEXT        0       None   0
2    2  deadlines  INTEGER        0       None   0


## Обработка и анализ данных

In [6]:
pd.io.sql.read_sql(
    """
    SELECT *
    FROM deadlines
    LIMIT 10;
    """,
    connection
)

Out[6]: 
   index      labs   deadlines
0      0    laba04  1587945599
1      1   laba04s  1587945599
2      2    laba05  1588550399
3      4    laba06  1590364799
4      5   laba06s  1590364799
5      3  project1  1589673599


## Обработка и анализ данных

In [7]:
df_min = pd.io.sql.read_sql(
    """
    SELECT t.uid, (strftime('%s', t.first_commit_ts) - d.deadlines)/3600.0 as diff
    FROM test as t
    JOIN deadlines as d ON t.labname=d.labs
    WHERE d.labs != 'project1'
    ORDER BY diff
    LIMIT 1
    """,
    connection
)
df_min

Out[7]: 
       uid     diff
0  user_30 -202.385


## Обработка и анализ данных

In [8]:
df_max = pd.io.sql.read_sql(
    """
    SELECT t.uid, (strftime('%s', t.first_commit_ts) - d.deadlines)/3600.0 as diff
    FROM test as t
    JOIN deadlines as d ON t.labname=d.labs
    WHERE d.labs != 'project1'
    ORDER BY diff DESC
    LIMIT 1
    """,
    connection
)
df_max

Out[8]: 
       uid    diff
0  user_25 -2.8675


## Обработка и анализ данных

In [9]:
df_avg = pd.io.sql.read_sql(
    """
    SELECT AVG((strftime('%s', t.first_commit_ts) - d.deadlines)/3600.0) as avg_diff
    FROM test as t
    JOIN deadlines as d ON t.labname=d.labs
    WHERE d.labs != 'project1'
    """,
    connection
)
df_avg

Out[9]: 
    avg_diff
0 -89.687841


## Обработка и анализ данных

In [10]:
views_diff = pd.io.sql.read_sql(
    """
    SELECT t.uid, AVG((strftime('%s', t.first_commit_ts) - d.deadlines)/3600.0) as avg_diff
    FROM test as t
    JOIN deadlines as d ON t.labname=d.labs
    WHERE d.labs != 'project1'
    GROUP BY t.uid
    """,
    connection
)
views_diff

Out[10]: 
        uid    avg_diff
0    user_1  -65.119778
1   user_10  -75.242444
2   user_14 -159.568796
3   user_17  -62.207667
4   user_18   -6.368148
5   user_19  -99.440417
6   user_21  -96.111181
7   user_25  -93.474944
8   user_28  -86.793833
9    user_3 -105.738222
10  user_30 -145.528681


## Обработка и анализ данных

In [11]:
pd.io.sql.read_sql(
    """
    SELECT uid, COUNT(*) AS pageviews
    FROM pageviews AS t
    GROUP BY t.uid;
    """,
    connection
)

Out[11]: 
        uid  pageviews
0   admin_0          3
1   admin_1         71
2   admin_2          1
3   admin_3         18
4    user_1         28
5   user_10         89
6   user_14        143
7   user_17         47
8   user_18          3
9   user_19         16
10  user_21         10
11  user_25        179
12  user_28        149
13   user_3        317
14  user_30          3
15   user_5          1
16   user_9          2


## Обработка и анализ данных

In [12]:
views_diff = pd.io.sql.read_sql(
    """
    SELECT a.uid, a.avg_diff, p.pageviews
    FROM (
        SELECT t.uid, AVG((strftime('%s', t.first_commit_ts) - d.deadlines)/3600.0) as avg_diff
        FROM test as t
        JOIN deadlines as d ON t.labname=d.labs
        WHERE d.labs != 'project1'
        GROUP BY t.uid
    ) AS a
    JOIN (
        SELECT uid, COUNT(*) AS pageviews
        FROM pageviews AS t
        GROUP BY t.uid
    ) AS p
        ON a.uid = p.uid;
    """
    ,connection
)
views_diff

Out[12]: 
        uid    avg_diff  pageviews
0    user_1  -65.119778         28
1   user_10  -75.242444         89
2   user_14 -159.568796        143
3   user_17  -62.207667         47
4   user_18   -6.368148          3
5   user_19  -99.440417         16
6   user_21  -96.111181         10
7   user_25  -93.474944        179
8   user_28  -86.793833        149
9    user_3 -105.738222        317
10  user_30 -145.528681          3


## Обработка и анализ данных

In [13]:
views_diff["avg_diff"].corr(views_diff["pageviews"])

Out[13]: np.float64(-0.27914346425426556)


## Закрытие соединения

In [14]:
connection.close()